In [4]:
#%%capture
#!wget "https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_2.12/3.5.0/spark-sql-kafka-0-10_2.12-3.5.0.jar"
#!wget "https://repo1.maven.org/maven2/org/apache/spark/spark-streaming-kafka-0-10_2.12/3.5.0/spark-streaming-kafka-0-10_2.12-3.5.0.jar"

!wget "https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_2.12/3.5.0/spark-sql-kafka-0-10_2.12-3.5.0.jar"
!wget "https://repo1.maven.org/maven2/org/apache/spark/spark-token-provider-kafka-0-10_2.12/3.5.0/spark-token-provider-kafka-0-10_2.12-3.5.0.jar"
!wget "https://repo1.maven.org/maven2/org/apache/commons/commons-pool2/2.11.1/commons-pool2-2.11.1.jar"

--2026-06-03 18:13:27--  https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_2.12/3.5.0/spark-sql-kafka-0-10_2.12-3.5.0.jar
Resolving repo1.maven.org (repo1.maven.org)... 104.18.19.12, 104.18.18.12, 2606:4700::6812:120c, ...
Connecting to repo1.maven.org (repo1.maven.org)|104.18.19.12|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 432335 (422K) [application/java-archive]
Saving to: ‘spark-sql-kafka-0-10_2.12-3.5.0.jar.5’

spark-sql-kafka-0-1 100%[===================>] 422.20K  --.-KB/s    in 0.03s   

2026-06-03 18:13:27 (12.8 MB/s) - ‘spark-sql-kafka-0-10_2.12-3.5.0.jar.5’ saved [432335/432335]

--2026-06-03 18:13:27--  https://repo1.maven.org/maven2/org/apache/spark/spark-token-provider-kafka-0-10_2.12/3.5.0/spark-token-provider-kafka-0-10_2.12-3.5.0.jar
Resolving repo1.maven.org (repo1.maven.org)... 104.18.19.12, 104.18.18.12, 2606:4700::6812:120c, ...
Connecting to repo1.maven.org (repo1.maven.org)|104.18.19.12|:443... connected.
HTTP req

In [1]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.5.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 pyspark-shell'

In [7]:
# Install OCR dependencies (pytesseract + Tesseract engine + Pillow)
%pip install pytesseract pillow
# On Debian/Ubuntu workers: sudo apt-get install -y tesseract-ocr
# For additional language packs: sudo apt-get install -y tesseract-ocr-tha  (Thai)
# For additional language packs: sudo apt-get install -y tesseract-ocr-jpn  (Japanese)  etc.

Note: you may need to restart the kernel to use updated packages.


In [ ]:
!sudo apt-get update && sudo apt-get install -y tesseract-ocr tesseract-ocr-tha tesseract-ocr-eng
import traceback
!tesseract --version

[sudo] password for jovyan: 

In [3]:
import pandas as pd
import os
import json
import base64
import time
from io import BytesIO
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName("OCR").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [4]:
# 1. Read Stream from Kafka
raw_stream = spark \
  .readStream \
  .format("kafka") \
  .option("kafka.bootstrap.servers", "192.168.1.141:8097") \
  .option("subscribe", "input") \
  .option("startingOffsets", "latest") \
  .load()

raw_stream.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [5]:
processed_data = raw_stream.select(
    col("key").cast(StringType()).alias("filename"),
    col("value").alias("png_data")
)

In [6]:
# -------------------------------------------------------
# OCR via Tesseract (open-source, no API key required)
# -------------------------------------------------------
import pytesseract
from PIL import Image
from pyspark.sql import Row

# Optional: set path to tesseract binary if not on PATH
# pytesseract.pytesseract.tesseract_cmd = r"/usr/bin/tesseract"

# Language(s) to recognise — use "+" to combine, e.g. "eng+tha"
#TESSERACT_LANG = "eng"
TESSERACT_LANG = "eng+tha"

# Page-segmentation mode: 3 = fully automatic (default)
TESSERACT_CONFIG = "--psm 3"

In [7]:
def call_tesseract_ocr(row: Row) -> tuple:
    """
    Perform OCR on a raw PNG/JPEG byte string using Tesseract.

    Parameters
    ----------
    row : Row
        PySpark Row with fields:
          - filename : str   (Kafka message key)
          - png_data : bytes (raw image bytes)

    Returns
    -------
    (filename: str, ocr_text: str)
    """
    filename = row["filename"]
    image_bytes = row["png_data"]

    # Decode bytes → PIL Image
    image = Image.open(BytesIO(image_bytes)).convert("RGB")

    # Run Tesseract OCR
    ocr_text = pytesseract.image_to_string(
        image,
        lang=TESSERACT_LANG,
        config=TESSERACT_CONFIG
    ).strip()

    # Wrap in JSON string to keep the same downstream schema
    json_string = json.dumps(ocr_text, ensure_ascii=False, indent=4)

    return filename, json_string

In [11]:
%pip install jiwer

Note: you may need to restart the kernel to use updated packages.


In [8]:
from jiwer import cer, wer

def evaluate_ocr_accuracy(ground_truth: str, ocr_text: str):
    """Calculate CER and WER between ground truth and OCR output."""
    character_error_rate = cer(ground_truth, ocr_text)
    word_error_rate = wer(ground_truth, ocr_text)
    return character_error_rate, word_error_rate

In [9]:
def process_batch(df, batch_id):
   try:
    """
    Executed for every micro-batch of the Structured Stream.
    Collects rows and runs Tesseract OCR on each image.
    """
    start_time_1 = time.time()

    # ⚠️ .collect() pulls all data to the driver.
    # Suitable for low-to-medium volumes; for high throughput
    # consider a Kafka Connect Sink or mapPartitions approach.
    rows = df.collect()

    results = []
    data_dict = {}
    filename = None

    for row in rows:
        print("row id =", batch_id)

        end_time_1 = time.time()
        time_use_1 = end_time_1 - start_time_1

        # ── OCR call (Tesseract, no external API) ──────────────────
        filename, ocr_text = call_tesseract_ocr(row)
        # ───────────────────────────────────────────────────────────

        start_time_2 = time.time()

        if filename is None:
            filename = f"image_{batch_id}"

        data_dict["filename"] = filename
        data_dict["message"] = ocr_text

        ground_truth_path = f"/home/jovyan/g{batch_id}.txt"
        print(f"ground_truth = {ground_truth_path}")

        if ground_truth_path and os.path.exists(ground_truth_path):
            with open(ground_truth_path, "r", encoding="utf-8") as f:
                ground_truth = (
                    json.load(f)
                    if ground_truth_path.endswith(".json")
                    else f.read().strip()
                )
            char_err, word_err = evaluate_ocr_accuracy(ground_truth, ocr_text)
            data_dict["char_err"] = char_err
            data_dict["word_err"] = word_err

        end_time_2 = time.time()
        time_use_2 = end_time_2 - start_time_2
        data_dict["time_usage"] = time_use_1 + time_use_2

        results.append((str(batch_id), data_dict))
        print(f"OCR Result: {results}")

    schema = StructType([
        StructField("filename", StringType(), False),
        StructField("ocr_text",  StringType(), False),
    ])

    if results:
        results_df = spark.createDataFrame(results, schema)

        batch_output_path = os.path.join("/home/jovyan/json", f"batch_{batch_id}")
        results_df.write \
            .format("json") \
            .mode("overwrite") \
            .save(batch_output_path)

   except Exception as e:
        print(f"❌ Error occurred in Python batch {batch_id}:")
        traceback.print_exc() # บรรทัดนี้จะพ่นตัวการที่แท้จริงออกมาใน Jupyter คอนโซลครับ
        raise e

In [10]:
query = processed_data.writeStream \
    .foreachBatch(process_batch) \
    .start()

query.awaitTermination()

row id = 1
❌ Error occurred in Python batch 1:


Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/pytesseract/pytesseract.py", line 275, in run_tesseract
    proc = subprocess.Popen(cmd_args, **subprocess_args())
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "/opt/conda/lib/python3.11/subprocess.py", line 1950, in _execute_child
    raise child_exception_type(errno_num, err_msg, err_filename)
FileNotFoundError: [Errno 2] No such file or directory: 'tesseract'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/tmp/ipykernel_22247/3986159093.py", line 25, in process_batch
    filename, ocr_text = call_tesseract_ocr(row)
                         ^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_22247/1805255287.py", line 23, in call_tesseract_ocr
    ocr_text = pytesseract.image_to_string(

StreamingQueryException: [STREAM_FAILED] Query [id = 64ee69a1-29a3-49ae-ba24-32f861a55a24, runId = 2aafa1c9-bfaa-4093-96a1-d79e1df868a7] terminated with exception: An exception was raised by the Python Proxy. Return Message: Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/pytesseract/pytesseract.py", line 275, in run_tesseract
    proc = subprocess.Popen(cmd_args, **subprocess_args())
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "/opt/conda/lib/python3.11/subprocess.py", line 1950, in _execute_child
    raise child_exception_type(errno_num, err_msg, err_filename)
FileNotFoundError: [Errno 2] No such file or directory: 'tesseract'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/

In [17]:
for q in spark.streams.active:
    q.stop()

In [ ]:
# Tesseract docs  : https://tesseract-ocr.github.io/
# pytesseract docs : https://github.com/madmaze/pytesseract
# Language packs   : apt-get install tesseract-ocr-<lang_code>